# Coleta de fundos Kinea — conteudos.xpi.com.br

**Por que `curl_cffi` e não `requests` puro:** a XP bloqueia por
*fingerprint* de TLS, não só por User-Agent. `curl_cffi` imita a
assinatura de um navegador real (Chrome).

**Design (revisado):** a reconstrução da base sempre lê o HTML salvo em
disco (`data/raw/html/`), nunca depende só do estado da memória do
Jupyter. Isso torna o notebook seguro de rodar em pedaços, em qualquer
ordem, quantas vezes for preciso — sem duplicar nem perder fundo.

## 1. Instalar dependências

In [1]:
%pip install curl_cffi beautifulsoup4 pandas --quiet

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. Imports e configuração

In [2]:
import csv
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_CSV = PROJECT_ROOT / "config" / "kinea_fund_urls.csv"
RAW_HTML_DIR = PROJECT_ROOT / "data" / "raw" / "html"
RAW_CSV_OUT = PROJECT_ROOT / "data" / "raw" / "universo_kinea_raw_scraped.csv"

print("Config:", CONFIG_CSV, "existe?", CONFIG_CSV.exists())

Config: /Users/julianamurakami/Downloads/case-kinea-bi/config/kinea_fund_urls.csv existe? True


## 3. Função de download

In [3]:
def baixar_html(url: str, timeout: int = 20):
    """Baixa o HTML impersonando Chrome. Retorna None (e avisa) se falhar,
    em vez de lançar exceção - um fundo com erro não derruba os outros."""
    try:
        resp = curl_requests.get(url, impersonate="chrome", timeout=timeout)
        if resp.status_code != 200:
            print(f"  [AVISO] {url} -> HTTP {resp.status_code}")
            return None
        return resp.text
    except Exception as exc:
        print(f"  [ERRO] {url} -> {exc}")
        return None

## 4. Extração de campos (rótulo → valor + tabelas de retorno/risco)

In [4]:
LABELS_FUNDO_ABERTO = {
    "CNPJ": "cnpj",
    "Aplicação Mínima": "aplicacao_minima",
    "Taxa de Administração (ao ano)": "taxa_administracao",
    "Taxa de Performance": "taxa_performance",
    "Cotização de Resgate": "cotizacao_resgate",
    "Liquidação de Resgate": "liquidacao_resgate",
    "Público Alvo": "publico_alvo",
    "Benchmark": "benchmark",
    "Classificação XP": "classificacao_xp",
    "Classificação CVM": "classificacao_cvm",
    "Gestor": "gestor",
    "Administrador": "administrador",
    "Custodiante": "custodiante",
    "Auditor": "auditor",
    "Risco": "risco_pontuacao_xp",
    "Data de Início": "data_inicio",
    "Rating Morningstar": "rating_morningstar",
    "Objetivo": "objetivo",
    "Tributação": "tributacao",
    "Volatilidade": "volatilidade_atual",
    "Drawdown": "drawdown_atual",
}

# NOTA: FII não tem campo "Objetivo" na própria página (estrutural, não é
# falha de extração - conferido direto no HTML). Não adicionar aqui.
LABELS_FII = {
    "Segmento": "segmento",
    "Taxa de administração": "taxa_administracao",
    "Público Alvo": "publico_alvo",
    "Dividendo Yield": "dividend_yield",
    "Valor Patrimonial": "valor_patrimonial",
    "Quantidade de Cotistas": "quantidade_cotistas",
}

CAMPOS_SO_NUMERICOS = {"risco_pontuacao_xp", "volatilidade_atual", "drawdown_atual"}


def _parece_numerico(texto: str) -> bool:
    return bool(re.fullmatch(r"-?\d+([.,]\d+)?%?", texto.strip()))


def extrair_tabela_risco_retorno(textos: list) -> dict:
    """Tabela 'Risco e Retorno' (fundo aberto): Rentabilidade, Volatilidade
    e Índice de Sharpe em 5 janelas comparáveis."""
    JANELAS = ["No Ano", "12 Meses", "24 Meses", "36 Meses", "Desde o Início"]
    resultado = {}
    if "Risco e Retorno" not in textos:
        return resultado
    idx = textos.index("Risco e Retorno")
    janelas_pagina = textos[idx + 1: idx + 1 + len(JANELAS)]
    cursor = idx + 1 + len(janelas_pagina)
    for metrica in ["Rentabilidade", "Volatilidade", "Índice de Sharpe"]:
        if metrica not in textos[cursor:]:
            continue
        j = textos.index(metrica, cursor)
        valores = textos[j + 1: j + 1 + len(janelas_pagina)]
        chave_metrica = metrica.lower().replace(" ", "_")
        for jan, val in zip(janelas_pagina, valores):
            resultado[f"{chave_metrica}__{jan.lower().replace(' ', '_')}"] = val
        cursor = j + 1 + len(janelas_pagina)
    return resultado


def extrair_tabela_retorno_fii(textos: list) -> dict:
    """Tabela de rentabilidade (FII): Fundo vs. benchmark em 6 janelas."""
    JANELAS = ["Dia", "Semana", "Mês", "3 Meses", "6 Meses", "No ano"]
    resultado = {}
    if "Rentabilidade" not in textos:
        return resultado
    idx = textos.index("Rentabilidade")
    j = idx + 1 + len(JANELAS)
    if j < len(textos) and textos[j] == "Fundo":
        valores_fundo = textos[j + 1: j + 1 + len(JANELAS)]
        for jan, val in zip(JANELAS, valores_fundo):
            resultado[f"rentab_fundo__{jan.lower().replace(' ', '_')}"] = val
        j2 = j + 1 + len(JANELAS)
        if j2 < len(textos) and textos[j2] in ("IBOV", "IFIX"):
            resultado["benchmark_retorno"] = textos[j2]
            valores_bench = textos[j2 + 1: j2 + 1 + len(JANELAS)]
            for jan, val in zip(JANELAS, valores_bench):
                resultado[f"rentab_benchmark__{jan.lower().replace(' ', '_')}"] = val
    return resultado


def extrair_campos(html: str, tipo_pagina: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    labels = LABELS_FII if tipo_pagina == "fii" else LABELS_FUNDO_ABERTO
    textos = [t for t in soup.stripped_strings]

    campos = {}
    for i, texto in enumerate(textos):
        for label, campo in labels.items():
            if texto.strip() != label or campo in campos:
                continue
            for j in range(i + 1, min(i + 4, len(textos))):
                candidato = textos[j].strip()
                if not candidato or candidato == label:
                    continue
                if campo in CAMPOS_SO_NUMERICOS and not _parece_numerico(candidato):
                    continue
                campos[campo] = candidato
                break

    if tipo_pagina == "fii":
        campos.update(extrair_tabela_retorno_fii(textos))
    else:
        campos.update(extrair_tabela_risco_retorno(textos))
    return campos

## 5. Adicionar fundos novos ao config (idempotente)

Registra fundos novos como `identificado_pendente`. Rodar de novo não
duplica quem já está no arquivo. É aqui que entra um fundo novo se
pedirem alteração ao vivo.

In [5]:
def adicionar_fundos_ao_config(novos_fundos: list) -> int:
    df_config = pd.read_csv(CONFIG_CSV)
    urls_existentes = set(df_config["url"])
    adicionados = 0
    for fundo in novos_fundos:
        if fundo["url"] not in urls_existentes:
            linha = {col: fundo.get(col) for col in df_config.columns}
            df_config = pd.concat([df_config, pd.DataFrame([linha])], ignore_index=True)
            adicionados += 1
            print(f"Adicionado: {fundo['nome_referencia']}")
        else:
            print(f"Já existia, pulei: {fundo['nome_referencia']}")
    if adicionados:
        df_config.to_csv(CONFIG_CSV, index=False, encoding="utf-8")
    print(f"\n{adicionados} fundo(s) novo(s) adicionado(s) ao config.")
    return adicionados


# Os 17 já estão todos no config - lista vazia por padrão. Se precisar
# adicionar um 18º fundo (ex: na arguição), preenche a lista abaixo.
adicionar_fundos_ao_config([])


0 fundo(s) novo(s) adicionado(s) ao config.


0

In [6]:
df_config = pd.read_csv(CONFIG_CSV)

alvo = df_config["nome_referencia"].isin([
    "Kinea Chronos FIM RL - Subclasse I",
    "Kinea Rendimentos Imobiliarios FII (KNCR11)",
])
df_config.loc[alvo, "status_confirmacao"] = "identificado_pendente"
df_config.to_csv(CONFIG_CSV, index=False, encoding="utf-8")

print(df_config[alvo][["nome_referencia", "status_confirmacao"]])

                               nome_referencia     status_confirmacao
0           Kinea Chronos FIM RL - Subclasse I  identificado_pendente
1  Kinea Rendimentos Imobiliarios FII (KNCR11)  identificado_pendente


## 6. Baixar HTML de todos os pendentes

Só baixa quem está `identificado_pendente`. Fundos já coletados
(`ficha_coletada`) não são re-baixados - o pipeline usa o HTML salvo em
disco (Seção 7), não depende de re-download.

In [7]:
with open(CONFIG_CSV, encoding="utf-8") as f:
    linhas_config = list(csv.DictReader(f))

pendentes = [l for l in linhas_config if l["status_confirmacao"] == "identificado_pendente"]
print(f"{len(pendentes)} fundo(s) pendente(s) a baixar.")

RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

for linha in pendentes:
    nome, url, tipo_pagina = linha["nome_referencia"], linha["url"], linha["tipo_pagina"]
    print(f"Baixando: {nome} ...")
    html = baixar_html(url)
    if html is not None:
        slug = re.sub(r"[^a-z0-9]+", "-", nome.lower()).strip("-")
        (RAW_HTML_DIR / f"{slug}.html").write_text(html, encoding="utf-8")
        print(f"  OK - salvo em {slug}.html")
    else:
        print(f"  FALHOU - tenta rodar essa célula de novo mais tarde")
    time.sleep(2)

print("\nDownload finalizado.")

2 fundo(s) pendente(s) a baixar.
Baixando: Kinea Chronos FIM RL - Subclasse I ...
  OK - salvo em kinea-chronos-fim-rl-subclasse-i.html
Baixando: Kinea Rendimentos Imobiliarios FII (KNCR11) ...
  OK - salvo em kinea-rendimentos-imobiliarios-fii-kncr11.html

Download finalizado.


## 7. Reconstruir a base a partir do HTML salvo em disco

**Fonte única da verdade**: sempre lê TODOS os HTMLs salvos em
`data/raw/html/` e reextrai com as funções da Seção 4 - nunca depende de
variável de memória de uma célula anterior. Roda quantas vezes quiser,
em qualquer ordem, sem risco de perder fundo já coletado.

In [8]:
resultados = []
faltando = []

for _, linha in pd.read_csv(CONFIG_CSV).iterrows():
    nome, tipo_pagina, url = linha["nome_referencia"], linha["tipo_pagina"], linha["url"]
    slug = re.sub(r"[^a-z0-9]+", "-", nome.lower()).strip("-")
    html_path = RAW_HTML_DIR / f"{slug}.html"

    if not html_path.exists():
        faltando.append(nome)
        continue

    campos = extrair_campos(html_path.read_text(encoding="utf-8"), tipo_pagina)
    resultados.append({
        "nome_referencia": nome,
        "url": url,
        "tipo_pagina": tipo_pagina,
        "source": "conteudos.xpi.com.br",
        "access_timestamp": linha.get("access_timestamp") or datetime.now(timezone.utc).isoformat(),
        "extraction_method": "curl_cffi_notebook_local",
        "erro": None,
        **campos,
    })

df_resultado = pd.DataFrame(resultados)
print(f"{len(df_resultado)} fundo(s) reconstruído(s) a partir do HTML salvo.")
if faltando:
    print(f"[AVISO] Sem HTML salvo para: {faltando} - rode a Seção 6 de novo pra esses.")

df_resultado.to_csv(RAW_CSV_OUT, index=False, encoding="utf-8")
print("Salvo em:", RAW_CSV_OUT)

17 fundo(s) reconstruído(s) a partir do HTML salvo.
Salvo em: /Users/julianamurakami/Downloads/case-kinea-bi/data/raw/universo_kinea_raw_scraped.csv


## 8. Juntar com fundos coletados só manualmente (se houver)

Se algum fundo ainda estiver só em `universo_kinea_raw_manual_original.csv`
(sem HTML próprio), entra aqui. Quem já tem HTML (Seção 7) tem prioridade
- versão mais rica.

In [9]:
raw_original_path = PROJECT_ROOT / "data" / "raw" / "universo_kinea_raw_manual_original.csv"
df_original = pd.read_csv(raw_original_path)

mapa_original_para_comum = {
    "nome_bruto": "nome_referencia", "cnpj_bruto": "cnpj", "ticker_bruto": "ticker",
    "categoria_bruta": "categoria_bruta", "publico_alvo_bruto": "publico_alvo",
    "taxa_administracao_bruta": "taxa_administracao", "taxa_performance_bruta": "taxa_performance",
    "aplicacao_minima_bruta": "aplicacao_minima", "cotizacao_resgate_bruta": "cotizacao_resgate",
}
df_original_renomeado = df_original.rename(columns=mapa_original_para_comum)

urls_ja_coletadas = set(df_resultado["url"])
df_original_filtrado = df_original_renomeado[
    ~df_original_renomeado["url"].isin(urls_ja_coletadas)
]

df_completo = pd.concat([df_original_filtrado, df_resultado], ignore_index=True, sort=False)
print(f"Total combinado: {len(df_completo)} fundos "
      f"({len(df_original_filtrado)} só-manuais + {len(df_resultado)} com HTML)")

caminho_completo = PROJECT_ROOT / "data" / "raw" / "universo_kinea_raw.csv"
df_completo.to_csv(caminho_completo, index=False, encoding="utf-8")
print("Base bruta completa salva:", caminho_completo)

Total combinado: 17 fundos (0 só-manuais + 17 com HTML)
Base bruta completa salva: /Users/julianamurakami/Downloads/case-kinea-bi/data/raw/universo_kinea_raw.csv


In [10]:
TEMPLATE_CAMPOS_ESPERADOS = {
    "fundo_aberto": ["objetivo", "benchmark", "tributacao", "risco_pontuacao_xp", "classificacao_xp", "classificacao_cvm"],
    "previdencia":  ["objetivo", "benchmark", "tributacao", "risco_pontuacao_xp", "classificacao_xp", "classificacao_cvm"],
    "fii":          ["segmento", "dividend_yield", "quantidade_cotistas", "valor_patrimonial"],
}

linhas = []
for _, f in df_completo.iterrows():
    esperados = TEMPLATE_CAMPOS_ESPERADOS.get(f["tipo_pagina"], [])
    presentes = [c for c in esperados if c in f.index and pd.notna(f[c]) and str(f[c]).strip() != ""]
    linhas.append({
        "fundo": f["nome_referencia"],
        "tipo_pagina": f["tipo_pagina"],
        "campos_esperados": len(esperados),
        "campos_presentes": len(presentes),
        "completude_pct": round(100 * len(presentes) / len(esperados), 1) if esperados else None,
        "campos_faltando": [c for c in esperados if c not in presentes],
    })

df_completude = pd.DataFrame(linhas).sort_values("completude_pct")
df_completude

,fundo,tipo_pagina,campos_esperados,campos_presentes,completude_pct,campos_faltando
11,Kinea Credito Agro FIAGRO FII (KNCA11),fii,4,2,50.0,"[quantidade_cotistas, valor_patrimonial]"
9,Kinea Infra FII (KDIF11),fii,4,2,50.0,"[quantidade_cotistas, valor_patrimonial]"
6,Kinea Alpes Prev XP Seg RF CP FICFI,previdencia,6,4,66.7,"[objetivo, classificacao_xp]"
0,Kinea Chronos FIM RL - Subclasse I,fundo_aberto,6,6,100.0,[]
14,Kinea Apolo FIF MM RL,fundo_aberto,6,6,100.0,[]
13,Kinea Andes FIF CIC RF CP LP RL,fundo_aberto,6,6,100.0,[]
12,Kinea High Yield CRI FII (KNHY11),fii,4,4,100.0,[]
10,Kinea Fundo de Fundos Imobiliarios FII (KFOF11),fii,4,4,100.0,[]
8,Kinea Indices de Precos FII (KNIP11),fii,4,4,100.0,[]
7,Kinea Renda Imobiliaria FII (KNRI11),fii,4,4,100.0,[]


## 9. Marcar como coletado no config

In [11]:
df_config = pd.read_csv(CONFIG_CSV)
df_config.loc[
    df_config["nome_referencia"].isin(urls_ja_coletadas), "status_confirmacao"
] = "ficha_coletada"
df_config.to_csv(CONFIG_CSV, index=False, encoding="utf-8")
print("Config atualizado.")

print("\nPróximo passo, no terminal:")
print("  python3 src/transformation/build_universe.py")
print("  python3 src/transformation/enrich_fii_cnpj.py")
print("  python3 src/quality/validate_universe.py")

Config atualizado.

Próximo passo, no terminal:
  python3 src/transformation/build_universe.py
  python3 src/transformation/enrich_fii_cnpj.py
  python3 src/quality/validate_universe.py
